# Profilage du dataset Geo DVF 2025 (PySpark)

Ce notebook profile le dataset **Geo DVF 2025** (transactions immobilières géolocalisées, ~3,7M lignes) pour deux objectifs :

1. **Comprendre la qualité de la donnée** : complétude, dédoublonnage des mutations, valeurs aberrantes, cohérence géographique.
2. **Produire les métriques de volume** qui permettront de décider du moteur de stockage de l'API (PostGIS vs DuckDB/Parquet).

On travaille en **PySpark** (exigence du module Big Data) et le code est volontairement transposable en nœuds Kedro.

> Prérequis : Java 17 actif (`sdk env` depuis `pipelines/dvf/`) et `pyspark>=3.5` installé.

In [1]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType, StructField, StructType

DATA_DIR = Path.cwd().parent / "data"
RAW_CSV_GZ = DATA_DIR / "01_raw" / "geo_dvf.csv.gz"
INTERMEDIATE_PARQUET = DATA_DIR / "02_intermediate" / "geo_dvf.parquet"

TYPES_BATIS = ["Maison", "Appartement"]

## Schéma explicite

On **n'utilise pas `inferSchema`** : sur 3,7M lignes c'est coûteux (2 passes) et instable. On fige le schéma, en gardant **les codes en `String`** pour préserver les zéros à gauche (codes INSEE commune/postal/département).

In [2]:
def _s(name):
    return StructField(name, StringType(), True)


def _d(name):
    return StructField(name, DoubleType(), True)


SCHEMA = StructType([
    _s("id_mutation"), _s("date_mutation"), _s("numero_disposition"),
    _s("nature_mutation"), _d("valeur_fonciere"), _s("adresse_numero"),
    _s("adresse_suffixe"), _s("adresse_nom_voie"), _s("adresse_code_voie"),
    _s("code_postal"), _s("code_commune"), _s("nom_commune"),
    _s("code_departement"), _s("ancien_code_commune"), _s("ancien_nom_commune"),
    _s("id_parcelle"), _s("ancien_id_parcelle"), _s("numero_volume"),
    _s("lot1_numero"), _d("lot1_surface_carrez"), _s("lot2_numero"),
    _d("lot2_surface_carrez"), _s("lot3_numero"), _d("lot3_surface_carrez"),
    _s("lot4_numero"), _d("lot4_surface_carrez"), _s("lot5_numero"),
    _d("lot5_surface_carrez"), _s("nombre_lots"), _s("code_type_local"),
    _s("type_local"), _d("surface_reelle_bati"), _d("nombre_pieces_principales"),
    _s("code_nature_culture"), _s("nature_culture"),
    _s("code_nature_culture_speciale"), _s("nature_culture_speciale"),
    _d("surface_terrain"), _d("longitude"), _d("latitude"),
])

In [3]:
spark = (
    SparkSession.builder.master("local[*]")
    .appName("profiling_geo_dvf")
    .config("spark.sql.shuffle.partitions", "64")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
spark.version

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/06/04 16:58:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


'3.5.8'

## Ingestion : gz → Parquet

Le `.gz` est **non-splittable** : Spark le lit sur un seul cœur. On le réécrit en **Parquet** dès maintenant pour débloquer le parallélisme sur toutes les analyses suivantes (et sur la future pipeline).

In [4]:
if not INTERMEDIATE_PARQUET.exists():
    raw = spark.read.csv(
        str(RAW_CSV_GZ), schema=SCHEMA, header=True, sep=",", encoding="UTF-8"
    )
    raw.repartition(32).write.mode("overwrite").parquet(str(INTERMEDIATE_PARQUET))

df = spark.read.parquet(str(INTERMEDIATE_PARQUET)).cache()
total = df.count()
print(f"{total:,} lignes")

26/06/04 16:58:22 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


3,714,829 lignes


## A. Complétude & cardinalités

Faits observés (run de référence) : `type_local` est vide pour **~40 %** des lignes (dépendances/terrains), `surface_reelle_bati` absente pour **~67 %** (idem), `valeur_fonciere` manquante pour **1,3 %**. Cardinalités : **1 314 051** mutations, **32 672** communes actives, **97** départements (DOM inclus).

In [5]:
exprs = []
for field in df.schema.fields:
    c = F.col(field.name)
    if isinstance(field.dataType, StringType):
        missing = F.sum(F.when(c.isNull() | (F.trim(c) == ""), 1).otherwise(0))
    else:
        missing = F.sum(F.when(c.isNull(), 1).otherwise(0))
    exprs.append((missing / total).alias(field.name))

missing_rate = df.select(exprs).collect()[0].asDict()
sorted(missing_rate.items(), key=lambda kv: -kv[1])[:12]


[
    ('ancien_id_parcelle', 0.9999978464688415),
    ('ancien_code_commune', 0.9999141279450549),
    ('ancien_nom_commune', 0.9999141279450549),
    ('lot5_surface_carrez', 0.999632553746081),
    ('lot4_surface_carrez', 0.999085287640427),
    ('numero_volume', 0.9976114647538286),
    ('lot5_numero', 0.997040778996826),
    ('lot3_surface_carrez', 0.9966324156508953),
    ('lot4_numero', 0.9941816971925221),
    ('lot3_numero', 0.9829060772380102),
    ('lot2_surface_carrez', 0.9691824307390731),
    ('code_nature_culture_speciale', 0.9556940575192021)
]

In [6]:
for col in ["id_mutation", "code_commune", "code_departement", "nature_mutation", "type_local"]:
    print(f"{col}: {df.select(F.countDistinct(col)).collect()[0][0]:,} valeurs distinctes")

id_mutation: 1,314,051 valeurs distinctes


code_commune: 32,672 valeurs distinctes


code_departement: 97 valeurs distinctes


nature_mutation: 6 valeurs distinctes


type_local: 4 valeurs distinctes


## B. Dédoublonnage des mutations

Une mutation (`id_mutation`) éclate en **N lignes** (une par couple parcelle × local). Faits observés : **2,83 lignes/mutation** en moyenne, médiane 2, **max 11 284** (vente en bloc massive). Point crucial : `valeur_fonciere` est **constante** sur toutes les lignes d'une mutation (0 mutation avec valeur non constante) → **on ne somme JAMAIS `valeur_fonciere`**.

In [7]:
per_mut = df.groupBy("id_mutation").agg(
    F.count("*").alias("n_lignes"),
    F.countDistinct("valeur_fonciere").alias("n_valeurs"),
)
per_mut.select(
    F.avg("n_lignes").alias("moyenne"),
    F.expr("percentile_approx(n_lignes, 0.5)").alias("mediane"),
    F.max("n_lignes").alias("max"),
).show()
print("Mutations avec valeur_fonciere non constante :",
      per_mut.filter(F.col("n_valeurs") > 1).count())

+-----------------+-------+-----+
|          moyenne|mediane|  max|
+-----------------+-------+-----+
|2.827005192340328|      2|11284|
+-----------------+-------+-----+



Mutations avec valeur_fonciere non constante : 0


## C. Valeurs aberrantes

`valeur_fonciere` présente des extrêmes massifs : **p1 = 0,1 €** (ventes symboliques/donations) et **p99 ≈ 695 M€** (ventes en bloc). Le bornage par quantiles est indispensable, mais il sera fait **sur le prix/m² par département × type** (cf. section D), pas sur la valeur brute.

In [8]:
print("valeur_fonciere nulle :", df.filter(F.col("valeur_fonciere").isNull()).count())
print("valeur_fonciere <= 0 :", df.filter(F.col("valeur_fonciere") <= 0).count())
print("quantiles p1/p50/p99 :", df.approxQuantile("valeur_fonciere", [0.01, 0.5, 0.99], 0.01))

valeur_fonciere nulle : 48420
valeur_fonciere <= 0 : 0


quantiles p1/p50/p99 : [0.1, 168000.0, 695000000.0]


## D. Prix au m² (méthodologie mono-bien)

`valeur_fonciere` est globale à la mutation → impossible de la ventiler proprement entre plusieurs biens. On calcule donc un prix/m² **uniquement sur les mutations "mono-bien bâti"** (exactement 1 Maison OU 1 Appartement), où l'attribution est non-ambiguë. Les multi-lots sont conservés pour le volume mais exclus du prix/m².

**Contrôle de cohérence** : la médiane Paris (dept 75) appartement ressort à **~9 687 €/m²**, parfaitement dans la fourchette attendue → la méthodologie est validée.

In [9]:
bati = df.filter(F.col("type_local").isin(TYPES_BATIS))
mono = (
    bati.groupBy("id_mutation").agg(F.count("*").alias("n_batis"))
    .filter(F.col("n_batis") == 1).select("id_mutation")
)
biens = (
    bati.join(mono, "id_mutation")
    .filter((F.col("surface_reelle_bati") > 0) & (F.col("valeur_fonciere") > 0))
    .withColumn("prix_m2", F.col("valeur_fonciere") / F.col("surface_reelle_bati"))
).cache()
print("Biens mono-bien exploitables :", biens.count())

biens.groupBy("type_local").agg(
    F.expr("percentile_approx(prix_m2, 0.5)").alias("mediane"),
    F.expr("percentile_approx(prix_m2, 0.25)").alias("p25"),
    F.expr("percentile_approx(prix_m2, 0.75)").alias("p75"),
).show()

Biens mono-bien exploitables : 753643


+-----------+------------------+------------------+------------------+
| type_local|           mediane|               p25|               p75|
+-----------+------------------+------------------+------------------+
|Appartement|3333.3333333333335|2208.3333333333335|            5000.0|
|     Maison|2166.6666666666665| 1402.439024390244|3157.8947368421054|
+-----------+------------------+------------------+------------------+



In [10]:
# Contrôle de cohérence Paris (dept 75)
biens.filter(
    (F.col("code_departement") == "75") & (F.col("type_local") == "Appartement")
).select(
    F.expr("percentile_approx(prix_m2, 0.5)").alias("mediane_paris_appart"),
    F.count("*").alias("n"),
).show()

+--------------------+-----+
|mediane_paris_appart|    n|
+--------------------+-----+
|   9686.153846153846|30053|
+--------------------+-----+



## E. Géographie & volume des agrégats (décision stockage)

**Conclusion stockage** : la table agrégée commune × type_bien fait **~40 600 lignes** — minuscule. Avec une maille purement administrative (pas de requête spatiale point-in-polygon), **DuckDB/Parquet en lecture seule suffit largement** ; PostGIS ne se justifierait que pour des requêtes géospatiales dynamiques, qui ne sont pas dans le périmètre.

Autres faits : **49 692** lignes sans coordonnées (~1,3 %), **1 596** communes avec moins de 5 transactions (→ flag de fiabilité, sans suppression).

In [11]:
print("Lignes sans lat/lon :",
      df.filter(F.col("longitude").isNull() | F.col("latitude").isNull()).count())
print("Communes actives :", df.select("code_commune").distinct().count())

volume_agg = (
    df.filter(F.col("type_local").isin(TYPES_BATIS))
    .select("code_commune", "type_local").distinct().count()
)
print("Volume agrégat commune x type :", volume_agg)

par_commune = df.groupBy("code_commune").count()
print("Communes < 5 transactions :", par_commune.filter(F.col("count") < 5).count())

Lignes sans lat/lon : 49692


Communes actives : 32672


Volume agrégat commune x type : 40579


Communes < 5 transactions : 1596


## Synthèse

| Métrique | Valeur | Conséquence pipeline |
|---|---|---|
| Lignes brutes | 3 714 830 | — |
| Mutations distinctes | 1 314 051 | dédoublonnage (2,83 lignes/mut) |
| `valeur_fonciere` constante/mut | oui (0 exception) | ne jamais sommer |
| Plus grosse mutation | 11 284 lignes | exclure du prix/m² (multi-lots) |
| Biens mono exploitables | 753 643 | base du prix/m² |
| Médiane Paris appart | ~9 687 €/m² | méthodologie validée |
| Communes actives | 32 672 | maille commune |
| Volume agg commune×type | ~40 600 | **DuckDB/Parquet suffit** |
| Communes < 5 tx | 1 596 | flag fiabilité |
| Sans coordonnées | 49 692 (1,3 %) | filtre géo |

In [12]:
spark.stop()